# 16 — Supplementary: TCR Diversity Metrics vs ICI Response

Compares five CDR3-based diversity metrics across Best Overall Response (BOR) categories
in ICI-treated patients, and shows their inter-correlations.

**Metrics**
- `cdr3_unique` — clonotype richness (unique CDR3 sequences)
- `cdr3_total`  — total CDR3 reads
- `shannon`     — Shannon entropy (diversity accounting for evenness)
- `evenness`    — Shannon entropy normalised by log(richness)
- `clonality`   — 1 − evenness (repertoire dominated by few clones)

**Panels**
- **A** — Boxplots of each metric by BOR (CR / PR / SD / PD)
- **B** — Correlogram (Spearman) of all five metrics

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(patchwork)
library(GGally)
library(ggpubr)
library(broom)
library(survival)
library(cowplot)


In [ ]:
args <- commandArgs(trailingOnly = FALSE)
script_path <- sub("--file=", "", args[grep("--file=", args)])
SCRIPT_DIR <- if (length(script_path) > 0) {
  normalizePath(dirname(script_path), mustWork = FALSE)
} else {
  getwd()
}
source(file.path(SCRIPT_DIR, "../helpers/map.r"))
source(file.path(SCRIPT_DIR, "../helpers/shortcuts.r"))
source(file.path(SCRIPT_DIR, "../helpers/config.r"))
source(file.path(SCRIPT_DIR, "../helpers/themes.r"))


## 1 — Data preparation

In [ ]:
# ICI-treated T-cell patients with BOR from cpi_ready.rds
cpi_raw <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(gp == "T-cell", cohort_broad == "ICI-Treated")

# Print column names to identify the BOR column
message("Columns in cpi_ready.rds (T-cell / ICI-Treated):")
print(names(cpi_raw))

# TODO: replace 'bor' below with the correct column name once identified
BOR_COL <- "response"

cpi <- cpi_raw |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::select(
    sampleId,
    bor       = response,
    dcb,
    osDays, osEvent,
    pfsEventDays, pfsEvent
  ) |>
  dplyr::mutate(
    bor = dplyr::recode(as.character(bor),
      "1" = "Responder", "0" = "Non-Responder",
      "Responder" = "Responder", "NonResponder" = "Non-Responder"),
    bor = factor(bor, levels = c("Non-Responder", "Responder"))
  )


In [ ]:
# Diversity metrics from cider_base.csv
diversity_raw <- readr::read_csv(
  paste0(CIDER_PROJ_DIR, "cider_base.csv"),
  show_col_types = FALSE
) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::select(sampleId, cdr3_unique, cdr3_total, shannon, evenness, clonality)

# Join with BOR data
div_df <- dplyr::inner_join(cpi, diversity_raw, by = "sampleId") |>
  dplyr::filter(!is.na(bor))

message("N = ", nrow(div_df), " patients with diversity + response data")
print(table(div_df$bor))


## 2 — Panel A: Boxplots by BOR

In [ ]:
METRICS <- c(
  "cdr3_unique" = "CDR3 Unique",
  "cdr3_total"  = "CDR3 Total Reads",
  "shannon"     = "Shannon Entropy",
  "clonality"   = "Clonality"
)

# Same visual style as every other BOR/response boxplot in this pipeline --
# helpers/themes.r's response_plot(), used by 1_response.ipynb and
# 17_response_by_read_confidence.ipynb: background jittered points (capped
# per group to avoid overplotting), boxplot with outliers hidden (the
# jitter already shows every point), a bold median label above each box,
# and an "n = " count label at the bottom. Not reused via response_plot()
# itself, since that helper dodges two dimensions (an x_col plus a
# responder fill) -- here `bor` is both the x-axis and the fill, a single
# dimension, so plain (non-dodged) jitter/labels are the direct equivalent.
make_boxplot <- function(metric, label) {
  use_log  <- metric %in% c("cdr3_unique", "cdr3_total")
  y_label  <- label

  df      <- div_df |> dplyr::filter(!is.na(.data[[metric]]))
  y_floor <- min(df[[metric]], na.rm = TRUE)

  # Per-group medians (shown above each box).
  med_labels <- df |>
    dplyr::group_by(bor) |>
    dplyr::summarise(med = median(.data[[metric]], na.rm = TRUE), .groups = "drop") |>
    dplyr::mutate(med_label = as.character(round(med, 2)))

  # Per-group counts (shown at the bottom of the plot, at the data's own
  # floor value -- generalises response_plot()'s "y = 0" placement, which
  # relies on its y_col's log10p1 transform having a zero floor; the plain
  # "log10" transform used here for cdr3_unique/cdr3_total has no zero, so
  # the observed minimum is used instead).
  count_labels <- df |>
    dplyr::group_by(bor) |>
    dplyr::summarise(n = dplyr::n(), .groups = "drop") |>
    dplyr::mutate(n_label = paste0("n = ", n))

  # Jitter data: cap at 150 points per response group to avoid overplotting
  # (same cap as response_plot()).
  jitter_data <- df |>
    dplyr::group_by(bor) |>
    dplyr::slice_sample(n = 150) |>
    dplyr::ungroup()

  df |>
    ggplot2::ggplot(ggplot2::aes(x = bor, y = .data[[metric]], fill = bor)) +
    ggplot2::geom_jitter(
      data     = jitter_data,
      ggplot2::aes(colour = bor),
      width = 0.15, alpha = 0.2, size = 0.5, show.legend = FALSE
    ) +
    ggplot2::scale_colour_manual(values = responder_colors, guide = "none") +
    ggplot2::geom_boxplot(alpha = 0.7, outlier.shape = NA) +
    ggplot2::geom_text(
      data        = med_labels,
      ggplot2::aes(x = bor, y = med, label = med_label),
      inherit.aes = FALSE,
      vjust = -0.4, size = 2.2, fontface = "bold", colour = COL_INK
    ) +
    ggplot2::geom_text(
      data        = count_labels,
      ggplot2::aes(x = bor, y = y_floor, label = n_label),
      inherit.aes = FALSE,
      vjust = 1.5, size = 2.1, colour = COL_NEGATIVE
    ) +
    ggpubr::stat_compare_means(
      ggplot2::aes(group = bor), method = "wilcox.test",
      label = "p.format", digits = 2, vjust = -1, size = 3
    ) +
    ggplot2::scale_fill_manual(values = responder_colors, guide = "none") +
    ggplot2::scale_y_continuous(
      trans  = if (use_log) "log10" else "identity",
      expand = ggplot2::expansion(mult = c(0.08, 0.18))
    ) +
    go_theme +
    ggplot2::scale_x_discrete(guide = ggplot2::guide_axis(n.dodge = 2)) +
    ggplot2::labs(x = NULL, y = y_label, title = y_label) +
    ggplot2::theme(
      plot.title  = ggplot2::element_text(size = 9, face = "bold", hjust = 0.5),
      axis.title.y = ggplot2::element_text(size = 8)
    )
}

# imap passes (value=label, name=column) — use .y for column, .x for label
box_list <- purrr::imap(METRICS, ~ make_boxplot(.y, .x))

p_boxes <- patchwork::wrap_plots(box_list, nrow = 1)

set_plot_dims(12, 4); p_boxes


## 3 — Panel B: Correlogram

In [ ]:
corr_df <- div_df |>
  dplyr::select(cdr3_unique, cdr3_total, shannon, clonality) |>
  dplyr::rename(
    "CDR3 Unique" = cdr3_unique, "CDR3 Total" = cdr3_total,
    "Shannon" = shannon, "Clonality" = clonality
  )

p_corr <- GGally::ggpairs(
  corr_df,
  xlab = "Value",
  ylab = "Value",
  upper = list(continuous = GGally::wrap("cor", method = "spearman", size = 3.5)),
  lower = list(continuous = GGally::wrap(
    "points", alpha = 0.15, size = 0.5, colour = COL_NEGATIVE
  )),
  diag  = list(continuous = GGally::wrap("densityDiag", colour = COL_NEGATIVE, fill = COL_NEGATIVE, alpha = 0.4))
) +
  go_theme +
  ggplot2::theme(
    strip.text  = ggplot2::element_text(size = 9),
    plot.margin = ggplot2::margin(t = 10, r = 10, b = 10, l = 10)
  ) +
  ggplot2::scale_x_continuous(n.breaks = 3) +
  ggplot2::scale_y_continuous(n.breaks = 3) +
  ggplot2::ggtitle("Diversity Metric Correlations")

set_plot_dims(7, 7); p_corr


## 5 — Assemble and save


## 4 — Panel C: Adjusted vs Unadjusted Logistic Regression Forest Plot


In [ ]:
# Scale all metrics (per-SD effects) for comparable coefficients
model_df <- div_df |>
  dplyr::mutate(
    resp_bin = as.integer(bor == "Responder"),
    dplyr::across(
      dplyr::all_of(names(METRICS)),
      ~ as.numeric(scale(.)),
      .names = "z_{.col}"
    )
  )

# Helper: fit one model and extract the metric term
fit_model <- function(metric, adjusted, endpoint) {
  z_var    <- paste0("z_", metric)
  adj_vars <- if (adjusted && metric != "cdr3_unique") c("z_cdr3_unique", z_var) else z_var

  fit <- tryCatch({
    if (endpoint == "BOR") {
      glm(stats::reformulate(adj_vars, "resp_bin"),
          data = model_df, family = binomial)
    } else if (endpoint == "DCB") {
      glm(stats::reformulate(adj_vars, "dcb"),
          data = dplyr::filter(model_df, !is.na(dcb)), family = binomial)
    } else if (endpoint == "PFS") {
      survival::coxph(
        stats::reformulate(adj_vars, "survival::Surv(pfsEventDays, pfsEvent)"),
        data = dplyr::filter(model_df, !is.na(pfsEventDays))
      )
    } else {
      survival::coxph(
        stats::reformulate(adj_vars, "survival::Surv(osDays, osEvent)"),
        data = dplyr::filter(model_df, !is.na(osDays))
      )
    }
  }, error = function(e) NULL)

  if (is.null(fit)) return(NULL)

  broom::tidy(fit, conf.int = TRUE, exponentiate = TRUE) |>
    dplyr::filter(term == z_var) |>
    dplyr::mutate(
      metric_label = METRICS[[metric]],
      metric_col   = metric,
      endpoint     = endpoint,
      model        = if (adjusted) "Adjusted for CDR3 Unique" else "Unadjusted"
    )
}

ENDPOINTS <- c("BOR", "DCB", "PFS", "OS")

forest_df <- purrr::map_dfr(names(METRICS), function(metric) {
  purrr::map_dfr(ENDPOINTS, function(ep) {
    rows <- dplyr::bind_rows(
      fit_model(metric, adjusted = FALSE, endpoint = ep),
      fit_model(metric, adjusted = TRUE,  endpoint = ep)
    )
    # Drop adjusted row for cdr3_unique (identical to unadjusted)
    dplyr::filter(rows, !(metric_col == "cdr3_unique" & model == "Adjusted for CDR3 Unique"))
  })
}) |>
  dplyr::mutate(
    metric_label = factor(metric_label, levels = rev(unname(METRICS))),
    endpoint     = factor(endpoint, levels = ENDPOINTS),
    # Adjusted first (bottom), Unadjusted last (top) in position_dodge
    model        = factor(model, levels = c("Adjusted for CDR3 Unique", "Unadjusted")),
    sig          = !is.na(p.value) & p.value < 0.05
  )

# Colour-by-endpoint was dropped: facet_wrap(~ endpoint) below already
# separates endpoints into their own panels, so an endpoint color legend was
# redundant. Every point now uses one flat colour (COL_INK); only shape
# (Model: Unadjusted/Adjusted) and alpha (significance) still vary.
p_forest <- ggplot2::ggplot(
  forest_df,
  ggplot2::aes(
    x      = estimate,
    y      = metric_label,
    xmin   = conf.low,
    xmax   = conf.high,
    shape  = model,
    alpha  = sig
  )
) +
  ggplot2::geom_vline(xintercept = 1, linetype = "dashed", colour = COL_NEGATIVE) +
  ggplot2::geom_pointrange(
    colour    = COL_INK,
    linewidth = 0.5, fatten = 3,
    position  = ggplot2::position_dodge(width = 0.6)
  ) +
  ggplot2::scale_shape_manual(
    values = c("Unadjusted" = 16, "Adjusted for CDR3 Unique" = 17),
    name   = "Model"
  ) +
  ggplot2::guides(
    # nrow = 1: flat/horizontal legend (both Model entries side by side)
    # rather than stacked.
    shape = ggplot2::guide_legend(nrow = 1, override.aes = list(colour = COL_INK))
  ) +
  ggplot2::scale_alpha_manual(
    values = c("TRUE" = 1.0, "FALSE" = 0.25), guide = "none"
  ) +
  ggplot2::facet_wrap(~ endpoint, nrow = 1, scales = "free_x") +
  go_theme +
  ggplot2::theme(
    strip.text      = ggplot2::element_text(face = "bold", size = 9),
    legend.position = "bottom"
  ) +
  ggplot2::labs(
    x     = "Effect Size (OR or HR, per SD)",
    y     = NULL,
    title = "Diversity Metrics vs ICI Endpoints: Unadjusted and Adjusted for CDR3 Unique"
  )

set_plot_dims(12, 4); p_forest

In [ ]:
# Capture ggpairs as a grob so cowplot can render it
p_corr_grob <- grid::grid.grabExpr(print(p_corr), wrap.grobs = TRUE)

panels <- cowplot::plot_grid(
  p_boxes, p_corr_grob, p_forest,
  ncol           = 1,
  rel_heights    = c(0.8, 1.32, 1),
  labels         = c("A", "B", "C"),
  label_size     = 11,
  label_fontface = "bold"
)

title <- cowplot::ggdraw() +
  cowplot::draw_label(
    "TCR Diversity Metrics and ICI Response",
    fontface = "bold", size = 13, hjust = 0.5
  )

combined <- cowplot::plot_grid(
  title, panels, ncol = 1, rel_heights = c(0.03, 1)
)

out <- paste0(CIDER_PROJ_DIR, "figs/supplement_diversity_metrics.png")
ggplot2::ggsave(out, combined, width = 9, height = 11, dpi = 300)
message("Saved: ", out)
